# Discrete-Time Signals and LTI Systems
This notebook covers the projects in this folder's README:
1. **Waveform visualization**: zooming, envelope and signal level.
2. **Frequency and phase analysis**, measured directly from the waveform (zero crossings and cross-correlation). Spectrum-based (FFT) analysis is the subject of `02_Frequency_Domain_Analysis`.
3. **Continuous and discrete-time signals**: from $x(t)$ to the sequence $x[n]$.
4. **Operations on sequences**: shifting, scaling and reversal.
5. **LTI systems**: linearity and time invariance.
6. **Impulse response and convolution**.
7. **Difference equations**: systems that feed back their own output, the basis of IIR filters.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from IPython import display
import ipywidgets as widgets
from ipywidgets import interactive
from audiodsp import add_noise, multi_sine, read_wav, sine, time_axis

%matplotlib inline
plt.style.use('seaborn-v0_8')
plt.rcParams['figure.figsize'] = (14, 5)

FS = 44100
DURATION = 2.0
t = time_axis(DURATION, FS)

# Test signals. Each shows a different time-domain feature.
attack_decay = (1 - np.exp(-t / 0.005)) * np.exp(-t / 0.5)
gate = (np.floor(t / 0.25) % 2 == 0).astype(float)
SIGNALS = {
    'Plucked note (attack and decay)': attack_decay * multi_sine([220, 440, 660, 880], DURATION, FS, amplitudes=[1, 0.5, 0.3, 0.2]) / 2,
    'Tremolo (4 Hz amplitude modulation)': sine(440, DURATION, FS) * (0.6 + 0.4 * np.sin(2 * np.pi * 4 * t)),
    'Tone bursts (on/off every 250 ms)': 0.8 * gate * sine(1000, DURATION, FS),
    'Chirp 100 Hz to 2 kHz': 0.8 * signal.chirp(t, 100, DURATION, 2000),
    'Noisy chord': add_noise(multi_sine([261.63, 329.63, 392.0], DURATION, FS, amplitudes=[0.3, 0.25, 0.2]), 0.05, np.random.default_rng(0)),
}

# To view your own recording, uncomment and set the path:
# x, fs_file = read_wav('my_recording.wav')
# assert fs_file == FS, 'resample or change FS to match the file'
# SIGNALS['My recording'] = x[:len(t)]

## 1. Waveform visualization
A waveform plots amplitude against time. At full length you see the **envelope**: the attack, decay and loudness changes. Zoomed in to a few milliseconds you see individual cycles and, at the deepest zoom, individual samples.

Signal level is measured in two ways:
- **Peak**: the largest absolute sample value.
- **RMS** (root mean square): $\sqrt{\frac{1}{N}\sum x[n]^2}$. RMS relates to perceived loudness and power. For a sine, RMS = peak / $\sqrt{2}$.

The **crest factor** is peak / RMS, in dB. A sine has a crest factor of 3 dB. Percussive sounds have much higher crest factors.

The envelope below is the short-time RMS over 10 ms frames.

In [ ]:
def frame_rms(x, frame):
    n_frames = len(x) // frame
    return np.sqrt(np.mean(x[:n_frames * frame].reshape(n_frames, frame)**2, axis=1))

def plot_waveform(name, start_s=0.1, window_ms=20.0, gain=1.0):
    x = gain * SIGNALS[name]
    frame = int(0.010 * FS)
    env = frame_rms(x, frame)
    env_t = (np.arange(len(env)) + 0.5) * frame / FS
    stop_s = start_s + window_ms / 1000
    zoom = (t >= start_s) & (t < stop_s)

    fig, axes = plt.subplots(2, 1, figsize=(14, 8))
    axes[0].plot(t, x, linewidth=0.5, label='waveform')
    axes[0].plot(env_t, env, 'r', linewidth=2, label='RMS envelope (10 ms)')
    axes[0].axvspan(start_s, stop_s, color='orange', alpha=0.3, label='zoom region')
    axes[0].set_title(name)
    axes[0].set_xlabel('Time (s)')
    axes[0].legend(loc='upper right')
    axes[1].plot(t[zoom] * 1000, x[zoom], '.-' if window_ms <= 5 else '-')
    axes[1].set_title(f'Zoom: {window_ms:g} ms from {start_s:.3f} s' + ('  (dots are samples)' if window_ms <= 5 else ''))
    axes[1].set_xlabel('Time (ms)')
    for ax in axes:
        ax.set_ylabel('Amplitude')
    plt.tight_layout()
    plt.show()

    peak, rms = np.max(np.abs(x)), np.sqrt(np.mean(x**2))
    print(f'Duration {len(x) / FS:.2f} s | peak {peak:.3f} | RMS {rms:.3f} | crest factor {20 * np.log10(peak / rms):.1f} dB')
    if peak > 1:
        print('Peak exceeds 1.0: this would clip when written as 16-bit audio.')
    display.display(display.Audio(np.clip(x, -1, 1), rate=FS, normalize=False))

display.display(interactive(
    plot_waveform,
    name=widgets.Dropdown(options=list(SIGNALS), description='Signal:', layout={'width': '450px'}),
    start_s=widgets.FloatSlider(value=0.1, min=0, max=DURATION - 0.05, step=0.005, description='Start (s):'),
    window_ms=widgets.SelectionSlider(options=[1, 2, 5, 10, 20, 50, 100, 500, 1000], value=20, description='Window (ms):'),
    gain=widgets.FloatSlider(value=1.0, min=0.1, max=3.0, step=0.1, description='Gain:'),
))

## 2. Frequency and phase in the time domain
### Frequency from zero crossings
A periodic waveform crosses zero going upwards once per cycle. The time between two upward crossings is one period $P$, and the frequency is $f = 1/P$. Measuring every period gives frequency **over time**, which tracks the rising pitch of a chirp.

Interpolating between the two samples either side of each crossing locates it to a fraction of a sample. Without interpolation, the estimate could only move in whole-sample steps (about 23 µs at 44.1 kHz).

This only works for signals with one dominant frequency. A chord or a tone with strong harmonics crosses zero several times per cycle.

In [ ]:
def upward_zero_crossings(x, fs):
    """Times (s) of upward zero crossings, linearly interpolated between samples."""
    i = np.where((x[:-1] < 0) & (x[1:] >= 0))[0]
    frac = -x[i] / (x[i + 1] - x[i])
    return (i + frac) / fs

chirp = SIGNALS['Chirp 100 Hz to 2 kHz']
crossings = upward_zero_crossings(chirp, FS)
periods = np.diff(crossings)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
zoom = t < 0.03
axes[0].plot(t[zoom] * 1000, chirp[zoom])
axes[0].plot(crossings[crossings < 0.03] * 1000, np.zeros(np.sum(crossings < 0.03)), 'ro', label='upward zero crossings')
axes[0].set_title('Start of the chirp')
axes[0].set_xlabel('Time (ms)')
axes[0].legend()
axes[1].plot(crossings[1:], 1 / periods, '.', markersize=3, label='1 / period between crossings')
axes[1].plot(t, 100 + (2000 - 100) * t / DURATION, 'k--', linewidth=1, label='true instantaneous frequency')
axes[1].set_title('Frequency over time from zero crossings')
axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('Frequency (Hz)')
axes[1].legend()
plt.tight_layout()
plt.show()

### Phase difference from cross-correlation
Phase only has meaning relative to something else: another signal at the same frequency, or a reference time. If $y(t) = \sin(2\pi f t + \varphi)$ and $x(t) = \sin(2\pi f t)$, then $y$ is $x$ shifted earlier by $\tau = \varphi / (2\pi f)$ seconds.

**Cross-correlation** slides one signal past the other and measures how well they match at each lag:

$$r_{yx}[k] = \sum_n y[n + k]\, x[n]$$

The best-matching lag within ±half a period gives $\tau$, and so the phase. The method tolerates noise because the noise does not line up with itself. With noise as strong as the signal, the error is typically within about ±15°. The resolution is one sample, which is $360 f / f_s$ degrees (3.6° at 440 Hz).

In [ ]:
def measure_phase(frequency=440, phase_deg=60, noise=0.0):
    rng = np.random.default_rng(1)
    n = int(0.1 * FS)
    x = add_noise(sine(frequency, 0.1, FS), noise, rng)
    y = add_noise(sine(frequency, 0.1, FS, phase=np.deg2rad(phase_deg)), noise, rng)

    r = signal.correlate(y, x, mode='full')
    lags = signal.correlation_lags(len(y), len(x), mode='full')
    half_period = int(FS / frequency / 2)
    search = np.abs(lags) <= half_period
    best_lag = lags[search][np.argmax(r[search])]
    # y[n + k] matches x[n] means y is x shifted earlier by k samples: phase = -360 f k / fs
    measured = (-360 * frequency * best_lag / FS + 180) % 360 - 180

    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
    show = np.arange(n) < int(3 * FS / frequency)
    axes[0].plot(np.arange(n)[show] / FS * 1000, x[show], label='x: reference')
    axes[0].plot(np.arange(n)[show] / FS * 1000, y[show], label=f'y: phase {phase_deg}°')
    axes[0].set_xlabel('Time (ms)')
    axes[0].set_title('Three periods of each signal')
    axes[0].legend(loc='upper right')
    axes[1].plot(lags[search] / FS * 1000, r[search])
    axes[1].axvline(best_lag / FS * 1000, color='r', linestyle='--', label=f'best lag = {best_lag} samples')
    axes[1].set_xlabel('Lag (ms)')
    axes[1].set_title('Cross-correlation within ±half a period')
    axes[1].legend()
    plt.tight_layout()
    plt.show()
    true = (phase_deg + 180) % 360 - 180
    print(f'True phase difference {true}°, measured {measured:.1f}° (resolution {360 * frequency / FS:.1f}°)')

display.display(interactive(
    measure_phase,
    frequency=widgets.IntSlider(value=440, min=50, max=2000, step=10, description='Freq (Hz):'),
    phase_deg=widgets.IntSlider(value=60, min=-180, max=180, step=5, description='Phase (°):'),
    noise=widgets.FloatSlider(value=0.0, min=0.0, max=2.0, step=0.1, description='Noise:'),
))

## 3. Continuous and discrete-time signals
A continuous-time signal $x(t)$ has a value at every instant. Sampling every $T = 1/f_s$ seconds turns it into a **sequence** of numbers indexed by an integer $n$:

$$x[n] = x(nT)$$

A sequence has no time unit of its own. $n$ only counts samples, and the sample rate links it back to seconds. Frequency becomes **normalized angular frequency** $\omega = 2\pi f / f_s$, in radians per sample. A sinusoid in discrete time is $x[n] = \cos(\omega n)$.

Two consequences follow from $n$ being an integer:
- $\omega$ and $\omega + 2\pi$ give exactly the same sequence. This is aliasing, covered in `00_Audio_Signals/sampling_and_quantization.ipynb`.
- A discrete sinusoid is periodic only if $\omega / 2\pi$ is a ratio of integers. A sampled 440 Hz tone at 44.1 kHz repeats exactly every 2205 samples (44100 / 440 = 2205 / 22), not every 100.2.

In [ ]:
def plot_discrete(frequency=440, fs=8000, n_samples=24):
    T = 1 / fs
    t_cont = np.linspace(0, n_samples * T, 2000)
    n = np.arange(n_samples)
    x_n = np.cos(2 * np.pi * frequency * n * T)
    omega = 2 * np.pi * frequency / fs

    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
    axes[0].plot(t_cont * 1000, np.cos(2 * np.pi * frequency * t_cont), color='0.6', label='x(t)')
    axes[0].stem(n * T * 1000, x_n, basefmt=' ', label='x(nT)')
    axes[0].set_xlabel('Time (ms)')
    axes[0].set_title(f'Continuous time: {frequency} Hz sampled at {fs} Hz')
    axes[0].legend(loc='upper right')
    axes[1].stem(n, x_n, basefmt=' ')
    axes[1].set_xlabel('Sample index n')
    axes[1].set_title(f'Discrete time: x[n] = cos({omega:.4f} n), ω = {omega / np.pi:.4f}π rad/sample')
    plt.tight_layout()
    plt.show()
    print('First 8 values of x[n]:', np.round(x_n[:8], 3))

display.display(interactive(
    plot_discrete,
    frequency=widgets.IntSlider(value=440, min=50, max=4000, step=10, description='Freq (Hz):'),
    fs=widgets.Dropdown(options=[2000, 4000, 8000, 16000, 44100], value=8000, description='fs (Hz):'),
    n_samples=widgets.IntSlider(value=24, min=8, max=100, description='Samples:'),
))

## 4. Operations on sequences
With $x[n]$ a short test sequence:
- **Delay** by $k$ samples: $y[n] = x[n - k]$ moves the sequence right (later). A negative $k$ advances it.
- **Scaling**: $y[n] = a\,x[n]$ changes the amplitude. $a < 0$ also inverts the polarity.
- **Time reversal**: $y[n] = x[-n]$ mirrors the sequence around $n = 0$.

In [ ]:
n = np.arange(-10, 11)
x_seq = np.where((n >= 0) & (n <= 5), 1.0 - n / 6, 0.0)   # ramp down over n = 0..5

def sequence_value(seq, idx):
    """seq[idx] on the index grid n, with zeros outside it."""
    out = np.zeros(len(idx))
    ok = (idx >= n[0]) & (idx <= n[-1])
    out[ok] = seq[idx[ok] - n[0]]
    return out

def plot_operations(delay=3, scale=1.5):
    ops = [
        ('x[n]', x_seq),
        (f'x[n - {delay}]', sequence_value(x_seq, n - delay)),
        (f'{scale} x[n]', scale * x_seq),
        ('x[-n]', sequence_value(x_seq, -n)),
    ]
    fig, axes = plt.subplots(1, 4, figsize=(16, 3.5), sharey=True)
    for ax, (title, y) in zip(axes, ops):
        ax.stem(n, y, basefmt=' ')
        ax.set_title(title)
        ax.set_xlabel('n')
    axes[0].set_ylim(min(-0.2, scale * 1.1), max(1.2, scale * 1.1))
    plt.tight_layout()
    plt.show()

display.display(interactive(
    plot_operations,
    delay=widgets.IntSlider(value=3, min=-5, max=5, description='Delay k:'),
    scale=widgets.FloatSlider(value=1.5, min=-2, max=2, step=0.1, description='Scale a:'),
))

## 5. Linear time-invariant (LTI) systems
A **system** turns an input sequence $x[n]$ into an output sequence $y[n]$. Two properties make a system much easier to analyze:
- **Linearity** (superposition): scaling and adding inputs scales and adds the outputs. If $x_1 \to y_1$ and $x_2 \to y_2$, then $a x_1 + b x_2 \to a y_1 + b y_2$.
- **Time invariance**: delaying the input delays the output by the same amount and changes nothing else. If $x[n] \to y[n]$, then $x[n-k] \to y[n-k]$.

A system with both properties is **LTI**. Filters, echoes and the acoustics of a room are LTI, to a very good approximation. Distortion is not linear. A tremolo, whose gain changes over time, is not time-invariant.

The cell below tests five systems with random inputs. Each error is the largest difference between the two sides of the property's equation, so an error near zero means the property holds.

In [ ]:
rng = np.random.default_rng(2)
tremolo_gain = 0.5 + 0.5 * np.sin(2 * np.pi * 5 * np.arange(4000) / FS)
SYSTEMS = {
    'Moving average (5 samples)': lambda x: np.convolve(x, np.ones(5) / 5)[:len(x)],
    'Echo (100 samples, 0.5)': lambda x: x + 0.5 * np.concatenate((np.zeros(100), x[:-100])),
    'Distortion: tanh(3x)': lambda x: np.tanh(3 * x),
    'Squarer: x^2': lambda x: x**2,
    'Tremolo: x[n] g[n]': lambda x: x * tremolo_gain[:len(x)],
}

def delay(x, k):
    return np.concatenate((np.zeros(k), x[:-k]))

x1, x2 = rng.normal(0, 0.3, 4000), rng.normal(0, 0.3, 4000)
a, b, k = 2.0, -0.7, 250
print(f'{"System":<28}{"Linearity error":>17}{"Time-invariance error":>23}   LTI?')
for name, system in SYSTEMS.items():
    linearity_error = np.max(np.abs(system(a * x1 + b * x2) - (a * system(x1) + b * system(x2))))
    time_invariance_error = np.max(np.abs(system(delay(x1, k)) - delay(system(x1), k)))
    is_lti = max(linearity_error, time_invariance_error) < 1e-9
    print(f'{name:<28}{linearity_error:>17.2e}{time_invariance_error:>23.2e}   {"yes" if is_lti else "no"}')

## 6. Impulse response and convolution
An LTI system is completely described by its **impulse response** $h[n]$: its output when the input is a single unit sample. For any input $x[n]$, the output is the **convolution**

$$y[n] = (x * h)[n] = \sum_k x[k]\, h[n - k]$$

To compute one output sample $y[n]$: flip $h$, slide it so it ends at position $n$, multiply it with $x$ sample by sample, and add up the products. Move the slider to step through the output.

In [ ]:
x_short = np.array([1.0, 2.0, 3.0, 1.0, -1.0])
h_short = np.array([1.0, 0.5, 0.25])
y_short = np.convolve(x_short, h_short)

def manual_convolution(x, h):
    y = np.zeros(len(x) + len(h) - 1)
    for i in range(len(y)):
        for k in range(len(x)):
            if 0 <= i - k < len(h):
                y[i] += x[k] * h[i - k]
    return y

assert np.allclose(manual_convolution(x_short, h_short), y_short)

def plot_convolution_step(n_out=2):
    k = np.arange(-len(h_short), len(x_short) + len(h_short))
    x_k = np.array([x_short[i] if 0 <= i < len(x_short) else 0 for i in k])
    h_flipped = np.array([h_short[n_out - i] if 0 <= n_out - i < len(h_short) else 0 for i in k])

    fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
    axes[0].stem(k, x_k, basefmt=' ')
    axes[0].set_title('x[k]')
    axes[1].stem(k, h_flipped, linefmt='C1-', markerfmt='C1o', basefmt=' ')
    axes[1].set_title(f'h[{n_out} - k]  (flipped, shifted to n = {n_out})')
    axes[2].stem(k, x_k * h_flipped, linefmt='C2-', markerfmt='C2o', basefmt=' ')
    axes[2].set_title(f'product, sum = {np.sum(x_k * h_flipped):g}')
    axes[3].stem(np.arange(len(y_short)), np.where(np.arange(len(y_short)) <= n_out, y_short, 0), basefmt=' ')
    axes[3].set_title(f'y[n] for n = 0..{n_out}')
    for ax in axes:
        ax.set_xlabel('k' if ax is not axes[3] else 'n')
    plt.tight_layout()
    plt.show()

print('x =', x_short, ' h =', h_short, ' y = x * h =', y_short)
display.display(interactive(
    plot_convolution_step,
    n_out=widgets.IntSlider(value=2, min=0, max=len(y_short) - 1, description='Output n:'),
))

### Convolution on audio
Each impulse response below is a simple audio processor. Convolving the plucked note with it applies the effect.
- **Moving average** of $M$ samples: $h[n] = 1/M$ for $n = 0..M-1$. Averaging neighbouring samples smooths out fast changes, so it acts as a low-pass filter (duller sound).
- **First difference**: $h = [1, -1]$, so $y[n] = x[n] - x[n-1]$. It keeps only changes, so it acts as a high-pass filter (thinner, brighter sound).
- **Echo**: $h[n] = \delta[n] + 0.5\,\delta[n - D]$ adds a copy delayed by $D$ samples (250 ms here) at half amplitude.

FIR filters (`03_Digital_Filters`) are this same operation with carefully designed impulse responses.

In [ ]:
note = SIGNALS['Plucked note (attack and decay)']
echo_delay = int(0.25 * FS)
impulse_responses = {
    'Moving average (M = 20)': np.ones(20) / 20,
    'First difference [1, -1]': np.array([1.0, -1.0]),
    'Echo (250 ms, 0.5)': np.concatenate(([1.0], np.zeros(echo_delay - 1), [0.5])),
}

fig, axes = plt.subplots(len(impulse_responses) + 1, 1, figsize=(14, 11), sharex=True)
axes[0].plot(t, note, linewidth=0.5)
axes[0].set_title('Input: plucked note')
print('Input:')
display.display(display.Audio(note, rate=FS))
for ax, (name, h) in zip(axes[1:], impulse_responses.items()):
    y = np.convolve(note, h)[:len(note)]
    ax.plot(t, y, linewidth=0.5)
    ax.set_title(f'{name}: impulse response length {len(h)}')
    print(f'{name}:')
    display.display(display.Audio(y, rate=FS))
axes[-1].set_xlabel('Time (s)')
plt.tight_layout()
plt.show()

## 7. Difference equations
Convolution computes each output from past *inputs*. A system can also reuse its own past *outputs*. The simplest example is the one-pole filter:

$$y[n] = x[n] + a\,y[n-1]$$

Feed in a single unit sample and the output is $1, a, a^2, a^3, \dots$, so the impulse response $h[n] = a^n$ never ends. A system like this is called **IIR** (infinite impulse response). With one multiplication per sample it does what a convolution could only do with an endless impulse response.
- $0 < a < 1$: the response decays smoothly. This is a simple low-pass filter (a "leaky integrator").
- $-1 < a < 0$: the response alternates in sign. This emphasizes high frequencies.
- $|a| \ge 1$: the response never decays, or grows without limit. The system is **unstable**.

`scipy.signal.lfilter(b, a, x)` runs any difference equation of this kind. `03_Digital_Filters` designs them.

In [ ]:
impulse = np.zeros(40)
impulse[0] = 1
fig, axes = plt.subplots(1, 5, figsize=(17, 3.5))
for ax, coeff in zip(axes, [0.5, 0.9, -0.9, 1.0, 1.05]):
    ax.stem(signal.lfilter([1.0], [1.0, -coeff], impulse), basefmt=' ')
    ax.set_title(f'a = {coeff}' + ('  (unstable)' if abs(coeff) >= 1 else ''))
    ax.set_xlabel('n')
plt.tight_layout()
plt.show()

# The recursion gives the same output as convolving with h[n] = a^n.
# a^400 is about 5e-19, so 400 terms of h are enough.
coeff = 0.9
x_test = rng.normal(0, 1, 2000)
recursive = signal.lfilter([1.0], [1.0, -coeff], x_test)
convolved = np.convolve(x_test, coeff ** np.arange(400))[:len(x_test)]
print(f'Max difference, recursion vs convolution with a^n: {np.max(np.abs(recursive - convolved)):.1e}')